# 7.3 Standard Library — itertools, functools & datetime

**Prerequisites:** 7.1 Python Module, 4.3 Generators, 4.4 Decorators  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- `itertools` — lazy building blocks for iteration
- `functools` — tools that operate on functions
- `datetime` — dates, times, durations and time zones
- **Aware vs naive datetimes**, and why naive ones cause production bugs
- Why `datetime.utcnow()` is deprecated in 3.12
- Practical patterns: batching, deduplication, expiry windows, retries

---

## Why these three

**7.1** tours `sys`, `math`, `random`, `collections`, `os`, `shutil`, `time`, `calendar` and
`pickle`. These three modules were missing, and they are arguably the ones you will reach for
most often in real work:

| Module | Answers |
|---|---|
| **`itertools`** | "How do I iterate over this in a way that doesn't load it all into memory?" |
| **`functools`** | "How do I cache / adapt / extend a function?" |
| **`datetime`** | "What time is it, and how do I not get this wrong?" |

Parts of `functools` appeared in **4.1** (`partial`, `cache`), **4.4** (`wraps`,
`singledispatch`) and **5.1** (`total_ordering`). This notebook pulls them together as a
reference.

---

## 1. `itertools`

Every `itertools` function returns a **lazy iterator** (see **4.3**). Nothing is computed
until you consume it, and nothing is stored. That is what makes it usable on data larger
than memory, or on infinite sequences.

### The ones worth memorising

| Function | Does | Typical use |
|---|---|---|
| `chain(a, b)` | Treat several iterables as one stream | Concatenating without copying |
| `chain.from_iterable(lists)` | Flatten one level | Flattening a list of lists |
| `islice(it, start, stop)` | Slice an iterator you cannot index | Taking the first N of a stream |
| `batched(it, n)` **(3.12+)** | Fixed-size chunks | Bulk API calls, DB inserts |
| `pairwise(it)` **(3.10+)** | Every adjacent pair | Deltas between readings |
| `groupby(it, key)` | Group **consecutive** equal items | Run-length encoding, sectioned reports |
| `accumulate(it)` | Running totals | Cumulative sums, balances |
| `product(a, b)` | Cartesian product | Replacing nested loops |
| `combinations` / `permutations` | Subsets / orderings | Test-case generation |
| `zip_longest(a, b)` | Zip without truncating | Padding uneven data |
| `takewhile` / `dropwhile` | Stop / skip while a predicate holds | Parsing headers |
| `count`, `cycle`, `repeat` | Infinite generators | IDs, round-robin, padding |

In [ ]:
import itertools
from itertools import (accumulate, batched, chain, combinations, count, cycle,
                       dropwhile, groupby, islice, pairwise, product, takewhile,
                       zip_longest)

# ---- chain: several iterables as one stream, no copying ----
weekday_logs = ["mon.log", "tue.log"]
weekend_logs = ["sat.log"]
print("chain      :", list(chain(weekday_logs, weekend_logs)))

nested = [[1, 2], [3, 4], [5]]
print("flatten    :", list(chain.from_iterable(nested)))

# ---- islice: slicing something you cannot index ----
def infinite_ids():
    n = 1000
    while True:
        yield f"ID-{n}"
        n += 1

print("islice     :", list(islice(infinite_ids(), 3)))

# ---- batched (3.12+): the chunking everyone hand-writes ----
records = list(range(1, 12))
print("\nbatched(4) :", [list(b) for b in batched(records, 4)])
print("  ^ note the last batch is short - batched never pads")

# ---- pairwise (3.10+): deltas between consecutive readings ----
temps = [20.1, 20.4, 21.9, 21.2, 23.0]
deltas = [round(b - a, 1) for a, b in pairwise(temps)]
print("\npairwise   :", deltas)
print("biggest jump:", max(deltas))

# ---- accumulate: running totals ----
daily_sales = [120, 80, 200, 50]
print("\naccumulate :", list(accumulate(daily_sales)))
print("running max:", list(accumulate(daily_sales, max)))

In [ ]:
from itertools import groupby, product, combinations, zip_longest, takewhile, dropwhile
from operator import itemgetter

# ---- groupby: groups CONSECUTIVE equal items, so SORT FIRST ----
events = [
    {"service": "api", "level": "ERROR"},
    {"service": "api", "level": "WARN"},
    {"service": "db", "level": "ERROR"},
    {"service": "api", "level": "INFO"},
]

print("⚠️ unsorted - 'api' appears twice:")
for key, group in groupby(events, key=itemgetter("service")):
    print(f"   {key}: {len(list(group))}")

print("\n✅ sorted first:")
events.sort(key=itemgetter("service"))
for key, group in groupby(events, key=itemgetter("service")):
    print(f"   {key}: {[g['level'] for g in group]}")

# Run-length encoding falls out of groupby for free
data = "aaabbbccd"
print("\nRLE        :", [(ch, len(list(g))) for ch, g in groupby(data)])


# ---- product: replaces nested loops ----
environments = ["dev", "prod"]
regions = ["eu", "us"]
print("\nproduct    :", [f"{e}-{r}" for e, r in product(environments, regions)])

# ---- combinations: every unordered pair, for test matrices ----
services = ["api", "db", "cache"]
print("pairs      :", list(combinations(services, 2)))

# ---- zip_longest: do not silently drop data ----
names = ["alice", "bob", "carol"]
scores = [90, 85]
print("\nzip        :", list(zip(names, scores)), " <- carol vanished")
print("zip_longest:", list(zip_longest(names, scores, fillvalue=0)))

# ---- takewhile / dropwhile: split a stream on a predicate ----
config_file = ["# header", "# generated", "host=localhost", "port=8080"]
print("\nheader     :", list(takewhile(lambda l: l.startswith("#"), config_file)))
print("body       :", list(dropwhile(lambda l: l.startswith("#"), config_file)))

---

## 2. `functools`

Tools that take a function and give you back a modified one.

| Tool | Does | Covered in |
|---|---|---|
| `cache` / `lru_cache` | Memoise results | **4.1** |
| `partial` | Freeze some arguments | **4.1** |
| `wraps` | Preserve metadata in a decorator | **4.4** |
| `singledispatch` | Dispatch on argument type | **4.4** |
| `total_ordering` | Derive all comparisons | **5.1** |
| `reduce` | Fold a sequence | **4.2** |
| `cached_property` | Compute once per instance | *new here* |

### `cached_property` — the one you have not met

A `@property` recomputes on every access. `@cached_property` computes once, then stores the
result **on the instance**, so later accesses are a plain attribute lookup.

Use it for expensive derived values on objects that live a while. ⚠️ It never invalidates —
if the underlying data changes, the cached value is stale.

In [ ]:
import time
from functools import cached_property, partial, cache, reduce


class LogFile:
    """Parsing is expensive; we only want to do it once per instance."""

    def __init__(self, lines: list[str]) -> None:
        self.lines = lines
        self.parse_count = 0

    @property
    def error_count_slow(self) -> int:
        self.parse_count += 1
        return sum(1 for line in self.lines if "ERROR" in line)

    @cached_property
    def error_count(self) -> int:
        self.parse_count += 1
        return sum(1 for line in self.lines if "ERROR" in line)


log = LogFile(["INFO ok", "ERROR db", "ERROR api", "WARN slow"] * 1000)

for _ in range(3):
    log.error_count_slow
print("@property      : parsed", log.parse_count, "times")

log.parse_count = 0
for _ in range(3):
    log.error_count
print("@cached_property: parsed", log.parse_count, "time")

# The cached value is stored as a normal instance attribute
print("\nin __dict__ now:", "error_count" in log.__dict__)

# ⚠️ It never invalidates
log.lines.append("ERROR new")
print("after adding an error:", log.error_count, " <- stale!")
del log.__dict__["error_count"]          # this is how you invalidate it
print("after invalidating   :", log.error_count)


# ---- partial: freeze arguments to build a specialised function ----
def send_request(method: str, url: str, timeout: float = 30.0) -> str:
    return f"{method} {url} (timeout={timeout})"

get = partial(send_request, "GET")
post_fast = partial(send_request, "POST", timeout=5.0)

print("\npartial:")
print("  ", get("/api/users"))
print("  ", post_fast("/api/orders"))
print("   frozen args:", get.args, "| keywords:", post_fast.keywords)


# ---- reduce: for genuinely custom folds ----
configs = [{"a": 1}, {"b": 2}, {"a": 99, "c": 3}]
merged = reduce(lambda acc, d: {**acc, **d}, configs, {})
print("\nreduce merge:", merged)
print("  (on 3.9+ you would just write: configs[0] | configs[1] | configs[2])")

---

## 3. `datetime`

Four classes, and knowing which one you need is most of the battle:

| Class | Represents | Example |
|---|---|---|
| `date` | A calendar day | `2024-03-15` |
| `time` | A time of day | `14:30:00` |
| `datetime` | Both | `2024-03-15 14:30:00` |
| `timedelta` | A **duration** | `3 days, 2:00:00` |

The key arithmetic rule:

```
datetime - datetime = timedelta
datetime + timedelta = datetime
timedelta + timedelta = timedelta
```

### 🔴 Aware vs naive — the bug that reaches production

A `datetime` is **naive** if it has no timezone information, and **aware** if it does.

```python
datetime(2024, 3, 15, 14, 30)                      # naive - 14:30 WHERE?
datetime(2024, 3, 15, 14, 30, tzinfo=timezone.utc) # aware - unambiguous
```

A naive datetime is a number without a unit. It works fine on your laptop, then breaks when
the server is in a different timezone, or when daylight saving shifts, or when two systems
compare timestamps. **You cannot even compare an aware and a naive datetime** — Python
raises `TypeError`.

> ### Version note — `utcnow()` is deprecated
> **`datetime.utcnow()` and `datetime.utcfromtimestamp()` are deprecated as of Python 3.12.**
> They return a **naive** datetime holding UTC values, which is the worst of both worlds: it
> looks like a local time but isn't one.
>
> | Don't | Do |
> |---|---|
> | `datetime.utcnow()` | `datetime.now(timezone.utc)` |
> | `datetime.utcfromtimestamp(ts)` | `datetime.fromtimestamp(ts, timezone.utc)` |
>
> **The rule:** store and compute in **UTC, aware**. Convert to local time only for display.

In [ ]:
from datetime import date, time, datetime, timedelta, timezone
import warnings

# ---- The four classes ----
d = date(2024, 3, 15)
t = time(14, 30)
dt = datetime(2024, 3, 15, 14, 30)
delta = timedelta(days=3, hours=2)

print("date     :", d, "|", type(d).__name__)
print("time     :", t)
print("datetime :", dt)
print("timedelta:", delta, "| total seconds:", delta.total_seconds())

# ---- Arithmetic ----
print("\ndt + delta       :", dt + delta)
print("difference       :", datetime(2024, 3, 20) - dt)
print("in days          :", (datetime(2024, 3, 20) - dt).days)


# ---- 🔴 Naive vs aware ----
naive = datetime(2024, 3, 15, 14, 30)
aware = datetime(2024, 3, 15, 14, 30, tzinfo=timezone.utc)

print("\nnaive :", naive, "| tzinfo:", naive.tzinfo)
print("aware :", aware, "| tzinfo:", aware.tzinfo)

try:
    naive < aware
except TypeError as exc:
    print("\ncomparing them:", exc)

# ---- The deprecated call ----
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    _ = datetime.utcnow()
    if caught:
        print("\ndatetime.utcnow():", caught[0].category.__name__)
        print("  ", str(caught[0].message).split(".")[0])
    else:
        print("\ndatetime.utcnow() is not yet deprecated on this version")

# ---- The correct way ----
now_utc = datetime.now(timezone.utc)
print("\ndatetime.now(timezone.utc):", now_utc.isoformat())
print("is aware                  :", now_utc.tzinfo is not None)

In [ ]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo          # stdlib since 3.9 - no pytz needed

# ---- Converting between zones ----
utc_now = datetime(2024, 3, 15, 9, 0, tzinfo=timezone.utc)

for zone in ["Asia/Kolkata", "Europe/London", "America/New_York"]:
    local = utc_now.astimezone(ZoneInfo(zone))
    print(f"  {zone:<20} {local:%Y-%m-%d %H:%M %Z}")

# ---- Daylight saving is handled for you ----
london = ZoneInfo("Europe/London")
winter = datetime(2024, 1, 15, 12, 0, tzinfo=london)
summer = datetime(2024, 7, 15, 12, 0, tzinfo=london)
print(f"\nLondon January: {winter:%H:%M %Z} (offset {winter.utcoffset()})")
print(f"London July   : {summer:%H:%M %Z} (offset {summer.utcoffset()})")


# ---- Parsing and formatting ----
raw = "2024-03-15 14:30:00"
parsed = datetime.strptime(raw, "%Y-%m-%d %H:%M:%S")
print("\nstrptime :", parsed)

# ISO 8601 is the format to use for machine-readable timestamps
iso = utc_now.isoformat()
print("isoformat:", iso)
print("round trip:", datetime.fromisoformat(iso) == utc_now)

print("\nformatting:")
for fmt, label in [("%Y-%m-%d", "ISO date"),
                   ("%d/%m/%Y", "day first"),
                   ("%B %d, %Y", "long month"),
                   ("%A", "weekday"),
                   ("%H:%M:%S %Z", "time + zone")]:
    print(f"  {label:<12} {utc_now:{fmt}}")


# ---- Practical: an expiry window ----
issued = datetime(2024, 3, 15, 9, 0, tzinfo=timezone.utc)
TTL = timedelta(minutes=15)
expires = issued + TTL

for offset in [0, 10, 20]:
    check = issued + timedelta(minutes=offset)
    valid = check < expires
    print(f"\n  at +{offset:>2}m: {'valid' if valid else 'EXPIRED'}"
          f" ({(expires - check).total_seconds() / 60:+.0f} min left)")

### Putting them together

A realistic pattern: process a stream of log records in batches, group them, and report
against a time window — using all three modules at once.

In [ ]:
from datetime import datetime, timedelta, timezone
from itertools import batched, groupby
from functools import reduce
from operator import itemgetter

START = datetime(2024, 3, 15, 9, 0, tzinfo=timezone.utc)

# A stream of records (imagine this coming from a file or an API)
records = [
    {"ts": START + timedelta(minutes=i), "service": svc, "level": lvl}
    for i, (svc, lvl) in enumerate([
        ("api", "ERROR"), ("api", "INFO"), ("db", "ERROR"), ("db", "ERROR"),
        ("api", "WARN"), ("cache", "INFO"), ("db", "INFO"), ("api", "ERROR"),
    ])
]

# ---- 1. Filter to a time window ----
window_end = START + timedelta(minutes=6)
in_window = [r for r in records if r["ts"] < window_end]
print(f"records in the first 6 minutes: {len(in_window)} of {len(records)}")

# ---- 2. Batch them, as you would for a bulk insert ----
print("\nbatched for bulk insert (size 3):")
for n, chunk in enumerate(batched(in_window, 3), start=1):
    stamps = [f"{r['ts']:%H:%M}" for r in chunk]
    print(f"  batch {n}: {stamps}")

# ---- 3. Group by service (sort first!) ----
by_service = sorted(records, key=itemgetter("service"))
print("\nerrors per service:")
for service, group in groupby(by_service, key=itemgetter("service")):
    items = list(group)
    errors = sum(1 for r in items if r["level"] == "ERROR")
    print(f"  {service:<6} {errors}/{len(items)} errors")

# ---- 4. Fold into a single summary ----
def tally(acc: dict, record: dict) -> dict:
    acc[record["level"]] = acc.get(record["level"], 0) + 1
    return acc

summary = reduce(tally, records, {})
print("\noverall:", summary)

span = max(r["ts"] for r in records) - min(r["ts"] for r in records)
print(f"time span: {span} ({span.total_seconds() / 60:.0f} minutes)")

---

## Common Mistakes & Pitfalls

1. 🔴 **Using `datetime.utcnow()`.** Deprecated in 3.12; it returns a *naive* datetime holding UTC values. Use `datetime.now(timezone.utc)`.
2. 🔴 **Mixing aware and naive datetimes.** Comparing them raises `TypeError`, and storing both in one system produces silent, timezone-shaped bugs.
3. **Calling `groupby` on unsorted data.** It groups only *consecutive* equal items, so an unsorted key appears in several groups. Sort by the same key first.
4. **Consuming an `itertools` result twice.** They are one-shot iterators — materialise with `list()` if you need more than one pass.
5. **Expecting `batched()` to pad the last chunk.** It does not; the final batch is short.
6. **`zip()` where the inputs may differ in length.** It silently truncates. Use `zip_longest`, or `strict=True` (**3.2.2**).
7. **`@cached_property` on mutable data.** It never invalidates — delete the entry from `instance.__dict__` to force a recompute.
8. **`@lru_cache` on a method.** It keeps `self` alive forever, leaking every instance.
9. **Using `timedelta` for calendar months.** Months are not a fixed duration; use `dateutil.relativedelta` or arithmetic on `date`.

## Best Practices

- Reach for `itertools` before hand-writing a chunking, grouping or windowing loop.
- Sort before `groupby`, always, using the same key function.
- Use `batched()` (3.12+) for bulk operations rather than manual index slicing.
- **Store and compute in UTC, aware.** Convert to local time only at the display edge.
- Use `zoneinfo` (stdlib, 3.9+) rather than the third-party `pytz`.
- Use ISO 8601 (`.isoformat()` / `fromisoformat()`) for machine-readable timestamps.
- Use `timedelta` for durations rather than raw seconds — the units document themselves.
- Use `@cache` for pure functions, `@cached_property` for expensive derived attributes.

## Practice Exercises

Try these before moving on.

1. Use `batched()` to send 1,000 records in chunks of 100, printing each chunk's size.
2. Use `pairwise()` to find the largest gap between consecutive timestamps in a log.
3. Group a list of orders by customer with `groupby` — first without sorting, to see it go wrong, then correctly.
4. Use `accumulate()` to produce a running account balance from a list of transactions.
5. Write a `@cached_property` that parses a config file, then demonstrate the staleness problem and fix it.
6. Convert a naive datetime to aware UTC, then display it in three time zones.
7. Show the `TypeError` from comparing an aware and a naive datetime, then fix it.
8. Write `is_expired(issued_at, ttl)` using aware datetimes and `timedelta`.
9. Replace a nested double loop somewhere in your earlier notebooks with `itertools.product`.